# 260903 0~4단계 HPPC fitting · C-rate validation

## 전체 연구 흐름

**Stage 0 — Ai2020 source correction**  
- Electrolyte diffusivity correction  
- Negative maximum concentration correction  

**Stage 1 — Experimental OCP + 1A endpoint-constrained stoichiometry**  
- Experimental Anode / Cathode OCP  
- Capacity-constrained stoichiometry span  
- Full-cell qOCV endpoint equation으로 `x0`, `y100` 결정  
- 0.5C / 1C / 2C dynamic baseline은 검증용으로만 사용  

**Stage 2A — BoL HPPC 7-parameter sensitivity / identifiability**  
- `kn, kp, Dsn, Dsp, brugg_n, brugg_p, brugg_s`
- Ai2020 nominal Bruggeman baseline  

**Stage 2B — Bruggeman = 1.5 고정 후 재분석**  
- `kn, kp, Dsn, Dsp` sensitivity / identifiability 재계산  

**Stage 3A / 3B — BoL HPPC parameter fitting**  
- HPPC만 calibration에 사용  
- TRF multi-start / Differential Evolution / Dual Annealing 비교  
- Global optimizer 결과를 TRF로 refinement  
- 알고리즘, seed/start, objective, nfev, RMSE, MAE, bound hit 기록  

**Stage 4 — 독립 C-rate validation**  
- HPPC fitting에는 사용하지 않은 0.5C / 1C / 2C Charge·Discharge 6조건  
- Time–Voltage RMSE / MAE / end-time error  
- Charge minimum anode potential 추적  

> Anode potential은 Stage별 내부상태 prediction으로 추적하며, 최종 절대값 검증은 추후 열화 데이터와의 간접 검증 또는 직접 실험 검증으로 열어둔다.


In [ ]:
!pip -q install pybamm==26.7.1.0 pandas openpyxl scipy matplotlib

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.interpolate import CubicSpline
from scipy.integrate import cumulative_trapezoid
from scipy.optimize import differential_evolution, dual_annealing, minimize, least_squares

import pybamm

warnings.filterwarnings("ignore", message="Workbook contains no default style")

ROOT = Path("/content")

plt.rcParams["figure.figsize"] = (7.5, 4.6)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

print("PyBaMM:", pybamm.__version__)

## 입력 파일

Colab `/content`에 아래 두 파일을 업로드한 뒤 실행한다.

1. `260902 Enertech셀 데이터 모음.xlsx`  
   - Half-cell OCP/GITT  
   - Full-cell C/20 qOCV  
   - 0.5C / 1C / 2C Charge·Discharge validation data

2. `260808 Eneterch셀 열화데이터 v1 6-6(5).xlsx`  
   - BoL HPPC raw data  
   - 본 notebook에서는 최초 HPPC, Cycle 1–10만 calibration에 사용

### HPPC fitting 데이터 해석
업로드된 raw export의 `record` 시트는 HPPC 구간에서 비교적 성긴 기록 간격을 갖는다.  
따라서 이번 feasibility fitting은 dense transient waveform 전체가 아니라 각 SOC block의 대표 전압점으로 수행한다.

- pre-pulse rest end
- 1C discharge pulse end (30 s)
- rest 30 s
- rest end (40 s)
- 0.75C charge pulse end (10 s)

총 10 SOC block × 5 point = 50개의 voltage target을 사용한다.


## 통합 입력 파일 업로드

필요 파일:
- `260902 Enertech셀 데이터 모음.xlsx`

원본 Neware/GITT 파일은 이 notebook에 다시 업로드하지 않는다.

통합 Excel의 각 데이터 시트 상단에는 원본 파일명과 실제 기록 기간이 별도로 기록되어 있으며,
실제 수치 데이터의 header는 9행부터 시작한다.


In [ ]:
BUNDLE = ROOT / "260902 Enertech셀 데이터 모음.xlsx"
print("Input data:", BUNDLE)

# Excel 내부 시트 확인
SHEETS = pd.ExcelFile(BUNDLE).sheet_names
print("Sheets:", SHEETS)

# 각 데이터 시트는 1~7행 source information, 8행 blank, 9행 header 구조
ANODE_LOWRATE_DF = pd.read_excel(
    BUNDLE, sheet_name="ANODE_LOWRATE", header=8
)

ANODE_GITT_DF = pd.read_excel(
    BUNDLE, sheet_name="ANODE_GITT", header=8
)

CATHODE_GITT_DF = pd.read_excel(
    BUNDLE, sheet_name="CATHODE_GITT", header=8
)

FULL_C20_DF = pd.read_excel(
    BUNDLE, sheet_name="FULLCELL_LOWRATE", header=8
)

DYNAMIC_DF = pd.read_excel(
    BUNDLE, sheet_name="DYNAMIC", header=8
)

# PARAMETER_PRIOR는 8행이 header
PRIOR_DF = pd.read_excel(
    BUNDLE, sheet_name="PARAMETER_PRIOR", header=7
)

#key=찾고 싶은 파라미터, 없을경우 None으로 처리, cast=가져온 값을 어떤 자료형으로 바꿀지
def prior(key, default=None, cast=float):
    row = PRIOR_DF[
        PRIOR_DF["parameter"].astype(str) == str(key)
    ]

    if row.empty:
        return default

    value = row.iloc[0]["value"]

    if cast is None:
        return value

    try:
        return cast(value)
    except Exception:
        return default


NOMINAL_CAPACITY_AH = prior("nominal_capacity_Ah")
Q_MEAS_BUNDLE = prior("q_meas_Ah")
T_REF = prior("temperature_K")
DISK_DIAMETER_M = prior("disk_diameter_m")

print("nominal capacity =", NOMINAL_CAPACITY_AH, "Ah")
print("bundle Q_MEAS    =", Q_MEAS_BUNDLE, "Ah")
print("T                =", T_REF, "K")
print("disk diameter    =", DISK_DIAMETER_M, "m")

print("\nLoaded rows")
print("ANODE_LOWRATE   :", len(ANODE_LOWRATE_DF))
print("ANODE_GITT      :", len(ANODE_GITT_DF))
print("CATHODE_GITT    :", len(CATHODE_GITT_DF))
print("FULLCELL        :", len(FULL_C20_DF))
print("DYNAMIC         :", len(DYNAMIC_DF))

## Stage 0. Ai2020 baseline source correction

In [ ]:
DE_MULTIPLIER = 1e-4
CSN_MAX_CORRECTED = 29700.0

# Python 관례
# *args   : 위치 기반(positional) 입력값들을 여러 개 받을 때 사용
# **kwargs: 이름이 지정된(keyword) 입력값들을 여러 개 받을 때 사용
def scale_parameter_function(func, factor):
    def wrapped(*args, **kwargs):
        return factor * func(*args, **kwargs)
    return wrapped


def make_source_corrected_ai2020():
    base = pybamm.ParameterValues("Ai2020")
    pv = base.copy()

    pv.update({
        "Electrolyte diffusivity [m2.s-1]":
            scale_parameter_function(
                base["Electrolyte diffusivity [m2.s-1]"],
                DE_MULTIPLIER,
            ),

        "Maximum concentration in negative electrode [mol.m-3]":
            CSN_MAX_CORRECTED,
    })

    return pv

AI2020_CORRECTED = make_source_corrected_ai2020()

print("=== Stage 0 correction ===")
print("De multiplier =",  DE_MULTIPLIER)
print(
    "c_s,n,max =",
    AI2020_CORRECTED[
        "Maximum concentration in negative electrode [mol.m-3]"
    ],
    "mol m^-3",
)

## Stage 1. Experimental OCP + stoichiometry window

In [ ]:
def extract_qv_branch(df, source_col, source, direction):

    mask = (
        (df[source_col] == source)
        & (df["direction"] == direction)
    )

    selected = df[mask]

    capacity = selected["Q_Ah"].to_numpy(dtype=float)
    voltage = selected["V_V"].to_numpy(dtype=float)

    return capacity, voltage


def extract_gitt_branch(df, source_col, source, pulse_direction):

    selected = df[
        (df[source_col] == source)
        & (df["pulse_direction"] == pulse_direction)
    ]

    capacity = selected["Qcum_Ah"].to_numpy(dtype=float)
    rest_voltage = selected["Vrest_V"].to_numpy(dtype=float)

    return np.c_[capacity, rest_voltage]


# Anode low-rate
# 원본: 260816 Anode GITT + OCP v1.xlsx / 260816 Anode GITT + OCP v2.xlsx
ANODE_V1_CHG_BRANCH = extract_qv_branch(
    ANODE_LOWRATE_DF, "source_version", "v1", "charge"
)

ANODE_V1_DCHG_BRANCH = extract_qv_branch(
    ANODE_LOWRATE_DF, "source_version", "v1", "discharge"
)

ANODE_V2_CHG_BRANCH = extract_qv_branch(
    ANODE_LOWRATE_DF, "source_version", "v2", "charge"
)

ANODE_V2_DCHG_BRANCH = extract_qv_branch(
    ANODE_LOWRATE_DF, "source_version", "v2", "discharge"
)


# Anode GITT
# 원본: 260816 Anode GITT + OCP v1.xlsx / 260816 Anode GITT + OCP v2.xlsx
ANODE_V1_GITT = {
    "CHG_BRANCH": extract_gitt_branch(
        ANODE_GITT_DF, "source_version", "v1", "charge_pulse"
    ),
    "DCHG_BRANCH": extract_gitt_branch(
        ANODE_GITT_DF, "source_version", "v1", "discharge_pulse"
    ),
}

ANODE_V2_GITT = {
    "CHG_BRANCH": extract_gitt_branch(
        ANODE_GITT_DF, "source_version", "v2", "charge_pulse"
    ),
    "DCHG_BRANCH": extract_gitt_branch(
        ANODE_GITT_DF, "source_version", "v2", "discharge_pulse"
    ),
}


# Cathode GITT
# 원본: 260816 Positive GITT 7-7.xlsx / 260816 Positive GITT 7-8.xlsx
CATHODE_77_GITT = {
    "CHG_BRANCH": extract_gitt_branch(
        CATHODE_GITT_DF, "source_channel", "7-7", "charge_pulse"
    ),
    "DCHG_BRANCH": extract_gitt_branch(
        CATHODE_GITT_DF, "source_channel", "7-7", "discharge_pulse"
    ),
}

CATHODE_78_GITT = {
    "CHG_BRANCH": extract_gitt_branch(
        CATHODE_GITT_DF, "source_channel", "7-8", "charge_pulse"
    ),
    "DCHG_BRANCH": extract_gitt_branch(
        CATHODE_GITT_DF, "source_channel", "7-8", "discharge_pulse"
    ),
}


# Full-cell C/20
# 원본: 260808 Enertech셀 초기 저율 데이터 v2.xlsx
FULLCELL_V2_CHG_BRANCH = extract_qv_branch(
    FULL_C20_DF, "source_version", "v2", "charge"
)

FULLCELL_V2_DCHG_BRANCH = extract_qv_branch(
    FULL_C20_DF, "source_version", "v2", "discharge"
)


print("Standardized Stage 1 data loaded.")

print(
    "Anode low-rate v1:",
    len(ANODE_V1_CHG_BRANCH[0]),
    len(ANODE_V1_DCHG_BRANCH[0])
)

print(
    "Anode low-rate v2:",
    len(ANODE_V2_CHG_BRANCH[0]),
    len(ANODE_V2_DCHG_BRANCH[0])
)

print(
    "Anode GITT v1:",
    len(ANODE_V1_GITT["CHG_BRANCH"]),
    len(ANODE_V1_GITT["DCHG_BRANCH"])
)

print(
    "Anode GITT v2:",
    len(ANODE_V2_GITT["CHG_BRANCH"]),
    len(ANODE_V2_GITT["DCHG_BRANCH"])
)

print(
    "Cathode GITT 7-7:",
    len(CATHODE_77_GITT["CHG_BRANCH"]),
    len(CATHODE_77_GITT["DCHG_BRANCH"])
)

print(
    "Cathode GITT 7-8:",
    len(CATHODE_78_GITT["CHG_BRANCH"]),
    len(CATHODE_78_GITT["DCHG_BRANCH"])
)

print(
    "Full-cell v2 C/20:",
    len(FULLCELL_V2_CHG_BRANCH[0]),
    len(FULLCELL_V2_DCHG_BRANCH[0])
)

In [ ]:
# ============================================================
# Cell 1. Capacity → Stoichiometry 변환
# ============================================================

FARADAY_CONSTANT = 96485.33212


# Maximum solid concentration
CSN_MAX = float(
    AI2020_CORRECTED[
        "Maximum concentration in negative electrode [mol.m-3]"
    ]
)

CSP_MAX = float(
    AI2020_CORRECTED[
        "Maximum concentration in positive electrode [mol.m-3]"
    ]
)


# Active material volume fraction
EPS_N = float(
    AI2020_CORRECTED[
        "Negative electrode active material volume fraction"
    ]
)

EPS_P = float(
    AI2020_CORRECTED[
        "Positive electrode active material volume fraction"
    ]
)


# Electrode thickness
LN = float(
    AI2020_CORRECTED[
        "Negative electrode thickness [m]"
    ]
)

LP = float(
    AI2020_CORRECTED[
        "Positive electrode thickness [m]"
    ]
)


# Ø10 mm half-cell electrode disk area
DISK_AREA = np.pi * (DISK_DIAMETER_M / 2.0)**2


# Half-cell electrode 기준 용량 [Ah]
QN_DISK_AH = (
    FARADAY_CONSTANT
    * CSN_MAX
    * EPS_N
    * LN
    * DISK_AREA
    / 3600
)

QP_DISK_AH = (
    FARADAY_CONSTANT
    * CSP_MAX
    * EPS_P
    * LP
    * DISK_AREA
    / 3600
)


def anode_stoich_axis(branch, direction):
    capacity, voltage = branch

    if direction == "lith":
        x = capacity / QN_DISK_AH

    else:
        span = capacity.max() / QN_DISK_AH
        x = span - capacity / QN_DISK_AH

    order = np.argsort(x)

    return x[order], voltage[order]


def cathode_stoich_axis(branch, direction):
    capacity = branch[:, 0]
    voltage = branch[:, 1]

    if direction == "lith":
        span = capacity.max() / QP_DISK_AH
        y = (1 - span) + capacity / QP_DISK_AH

    else:
        y = 1 - capacity / QP_DISK_AH

    order = np.argsort(y)

    return y[order], voltage[order]


# ------------------------------------------------------------
# Anode stoichiometry 변환
# Graphite/Li half-cell
# DCHG → lithiation   (Li insertion, x 증가)
# CHG  → delithiation (Li extraction, x 감소)
#
# 원본:
# 260816 Anode GITT + OCP v1.xlsx
# 260816 Anode GITT + OCP v2.xlsx
# ------------------------------------------------------------

ANODE_V1_LITH = anode_stoich_axis(
    ANODE_V1_DCHG_BRANCH,
    "lith"
)

ANODE_V1_DELITH = anode_stoich_axis(
    ANODE_V1_CHG_BRANCH,
    "delith"
)

ANODE_V2_LITH = anode_stoich_axis(
    ANODE_V2_DCHG_BRANCH,
    "lith"
)

ANODE_V2_DELITH = anode_stoich_axis(
    ANODE_V2_CHG_BRANCH,
    "delith"
)


# ------------------------------------------------------------
# Cathode stoichiometry 변환
# LCO/Li half-cell
# DCHG pulse → lithiation   (Li insertion, y 증가)
# CHG pulse  → delithiation (Li extraction, y 감소)
#
# 원본:
# 260816 Positive GITT 7-7.xlsx
# 260816 Positive GITT 7-8.xlsx
# ------------------------------------------------------------

CATHODE_77_LITH = cathode_stoich_axis(
    CATHODE_77_GITT["DCHG_BRANCH"],
    "lith"
)

CATHODE_77_DELITH = cathode_stoich_axis(
    CATHODE_77_GITT["CHG_BRANCH"],
    "delith"
)

CATHODE_78_LITH = cathode_stoich_axis(
    CATHODE_78_GITT["DCHG_BRANCH"],
    "lith"
)

CATHODE_78_DELITH = cathode_stoich_axis(
    CATHODE_78_GITT["CHG_BRANCH"],
    "delith"
)


print("QN disk capacity =", QN_DISK_AH, "Ah")
print("QP disk capacity =", QP_DISK_AH, "Ah")

In [ ]:
# ============================================================
# Cell 2. OCP / Stoichiometry 변환 결과 확인
# ============================================================


# ------------------------------------------------------------
# 1. Anode 원본 Capacity-Voltage curve
# ------------------------------------------------------------

plt.figure(figsize=(8, 5.5))

plt.plot(
    ANODE_V1_CHG_BRANCH[0],
    ANODE_V1_CHG_BRANCH[1],
    label="v1 CHG"
)

plt.plot(
    ANODE_V1_DCHG_BRANCH[0],
    ANODE_V1_DCHG_BRANCH[1],
    label="v1 DCHG"
)

plt.plot(
    ANODE_V2_CHG_BRANCH[0],
    ANODE_V2_CHG_BRANCH[1],
    label="v2 CHG"
)

plt.plot(
    ANODE_V2_DCHG_BRANCH[0],
    ANODE_V2_DCHG_BRANCH[1],
    label="v2 DCHG"
)

plt.xlabel("Capacity [Ah]")
plt.ylabel("Voltage [V vs. Li/Li+]")
plt.title("Anode half-cell: Original Q-V curves")

plt.legend()
plt.grid()

plt.figtext(
    0.5,
    -0.02,
    "Source: 260816 Anode GITT + OCP v1.xlsx / "
    "260816 Anode GITT + OCP v2.xlsx",
    ha="center",
    fontsize=9
)

plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 2. Anode Stoichiometry-Voltage curve
# ------------------------------------------------------------

plt.figure(figsize=(8, 5.5))

plt.plot(
    ANODE_V1_LITH[0],
    ANODE_V1_LITH[1],
    label="v1 lith (DCHG)"
)

plt.plot(
    ANODE_V1_DELITH[0],
    ANODE_V1_DELITH[1],
    label="v1 delith (CHG)"
)

plt.plot(
    ANODE_V2_LITH[0],
    ANODE_V2_LITH[1],
    label="v2 lith (DCHG)"
)

plt.plot(
    ANODE_V2_DELITH[0],
    ANODE_V2_DELITH[1],
    label="v2 delith (CHG)"
)

plt.xlabel("Anode stoichiometry x")
plt.ylabel("Voltage [V vs. Li/Li+]")
plt.title(
    "Anode half-cell: Stoichiometry-converted OCP curves"
)

plt.legend()
plt.grid()

plt.figtext(
    0.5,
    -0.04,
    "Source: 260816 Anode GITT + OCP v1.xlsx / "
    "260816 Anode GITT + OCP v2.xlsx\n"
    "Graphite/Li: DCHG → lithiation, CHG → delithiation",
    ha="center",
    fontsize=9
)

plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 3. Cathode 원본 GITT Capacity-Rest Voltage curve
# ------------------------------------------------------------

plt.figure(figsize=(8, 5.5))

plt.plot(
    CATHODE_77_GITT["CHG_BRANCH"][:, 0],
    CATHODE_77_GITT["CHG_BRANCH"][:, 1],
    marker="o",
    label="7-7 CHG pulse"
)

plt.plot(
    CATHODE_77_GITT["DCHG_BRANCH"][:, 0],
    CATHODE_77_GITT["DCHG_BRANCH"][:, 1],
    marker="o",
    label="7-7 DCHG pulse"
)

plt.plot(
    CATHODE_78_GITT["CHG_BRANCH"][:, 0],
    CATHODE_78_GITT["CHG_BRANCH"][:, 1],
    marker="o",
    label="7-8 CHG pulse"
)

plt.plot(
    CATHODE_78_GITT["DCHG_BRANCH"][:, 0],
    CATHODE_78_GITT["DCHG_BRANCH"][:, 1],
    marker="o",
    label="7-8 DCHG pulse"
)

plt.xlabel("Cumulative capacity [Ah]")
plt.ylabel("Rest voltage [V vs. Li/Li+]")
plt.title("Cathode half-cell: Original GITT OCP points")

plt.legend()
plt.grid()

plt.figtext(
    0.5,
    -0.02,
    "Source: 260816 Positive GITT 7-7.xlsx / "
    "260816 Positive GITT 7-8.xlsx",
    ha="center",
    fontsize=9
)

plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 4. Cathode Stoichiometry-Voltage curve
# ------------------------------------------------------------

plt.figure(figsize=(8, 5.5))

plt.plot(
    CATHODE_77_LITH[0],
    CATHODE_77_LITH[1],
    marker="o",
    label="7-7 lith (DCHG)"
)

plt.plot(
    CATHODE_77_DELITH[0],
    CATHODE_77_DELITH[1],
    marker="o",
    label="7-7 delith (CHG)"
)

plt.plot(
    CATHODE_78_LITH[0],
    CATHODE_78_LITH[1],
    marker="o",
    label="7-8 lith (DCHG)"
)

plt.plot(
    CATHODE_78_DELITH[0],
    CATHODE_78_DELITH[1],
    marker="o",
    label="7-8 delith (CHG)"
)

plt.xlabel("Cathode stoichiometry y")
plt.ylabel("Voltage [V vs. Li/Li+]")
plt.title(
    "Cathode half-cell: Stoichiometry-converted OCP curves"
)

plt.legend()
plt.grid()

plt.figtext(
    0.5,
    -0.04,
    "Source: 260816 Positive GITT 7-7.xlsx / "
    "260816 Positive GITT 7-8.xlsx\n"
    "LCO/Li: DCHG → lithiation, CHG → delithiation",
    ha="center",
    fontsize=9
)

plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 5. Stoichiometry range 확인
# ------------------------------------------------------------

print("=== Anode stoichiometry range ===")

print(
    "v1 lith   :",
    ANODE_V1_LITH[0].min(),
    "to",
    ANODE_V1_LITH[0].max()
)

print(
    "v1 delith :",
    ANODE_V1_DELITH[0].min(),
    "to",
    ANODE_V1_DELITH[0].max()
)

print(
    "v2 lith   :",
    ANODE_V2_LITH[0].min(),
    "to",
    ANODE_V2_LITH[0].max()
)

print(
    "v2 delith :",
    ANODE_V2_DELITH[0].min(),
    "to",
    ANODE_V2_DELITH[0].max()
)


print("\n=== Cathode stoichiometry range ===")

print(
    "7-7 lith   :",
    CATHODE_77_LITH[0].min(),
    "to",
    CATHODE_77_LITH[0].max()
)

print(
    "7-7 delith :",
    CATHODE_77_DELITH[0].min(),
    "to",
    CATHODE_77_DELITH[0].max()
)

print(
    "7-8 lith   :",
    CATHODE_78_LITH[0].min(),
    "to",
    CATHODE_78_LITH[0].max()
)

print(
    "7-8 delith :",
    CATHODE_78_DELITH[0].min(),
    "to",
    CATHODE_78_DELITH[0].max()
)


# ------------------------------------------------------------
# 6. 여러 OCP curve의 공통 범위에서 평균 curve 생성
# ------------------------------------------------------------

def mean_overlap(curves, n=3000):

    lower_limit = max(
        x.min() for x, _ in curves
    )

    upper_limit = min(
        x.max() for x, _ in curves
    )

    assert upper_limit > lower_limit, (
        "OCP curve common overlap이 없습니다."
    )

    grid = np.linspace(
        lower_limit,
        upper_limit,
        n
    )

    values = np.vstack([
        np.interp(grid, x, y)
        for x, y in curves
    ])

    return grid, values.mean(axis=0)

In [ ]:
# ============================================================
# Experimental OCP 구성 + Ai2020 비교
# ============================================================


# ------------------------------------------------------------
# 1. Experimental Anode OCP
#
# 현재 backbone:
# v1 / v2 low-rate OCP
# ------------------------------------------------------------

X_LITH, UN_LITH = mean_overlap([
    ANODE_V1_LITH,
    ANODE_V2_LITH,
])

X_DELITH, UN_DELITH = mean_overlap([
    ANODE_V1_DELITH,
    ANODE_V2_DELITH,
])


X_MIN = max(
    X_LITH.min(),
    X_DELITH.min()
)

X_MAX = min(
    X_LITH.max(),
    X_DELITH.max()
)

X_EXP = np.linspace(
    X_MIN,
    X_MAX,
    3000
)


UN_EXP = 0.5 * (
    np.interp(
        X_EXP,
        X_LITH,
        UN_LITH
    )
    +
    np.interp(
        X_EXP,
        X_DELITH,
        UN_DELITH
    )
)


# ------------------------------------------------------------
# 2. Experimental Cathode OCP
#
# 현재 backbone:
# GITT 7-7 / 7-8 rest-end quasi-OCP
# ------------------------------------------------------------

Y_EXP, UP_EXP = mean_overlap([
    CATHODE_77_LITH,
    CATHODE_78_LITH,
    CATHODE_77_DELITH,
    CATHODE_78_DELITH,
])


# ------------------------------------------------------------
# 3. Ai2020 OCP 불러오기
# ------------------------------------------------------------

PYBAMM_DIR = Path(
    pybamm.__file__
).resolve().parent

DATA_DIR = (
    PYBAMM_DIR
    / "input"
    / "parameters"
    / "lithium_ion"
    / "data"
)


def read_ai2020_ocp(filename):

    data = np.genfromtxt(
        DATA_DIR / filename,
        delimiter=","
    )

    data = data[
        np.isfinite(data).all(axis=1)
    ]

    return data[:, 0], data[:, 1]


X_AI, UN_AI = read_ai2020_ocp(
    "graphite_ocp_Enertech_Ai2020.csv"
)

Y_AI, UP_AI = read_ai2020_ocp(
    "lico2_ocp_Ai2020.csv"
)


# ------------------------------------------------------------
# 4. Numerical OCP 함수
#
# 뒤의 stoichiometry window fitting에서 사용
# ------------------------------------------------------------

UN_AI_SPLINE = CubicSpline(
    X_AI,
    UN_AI,
    extrapolate=True
)

UP_AI_SPLINE = CubicSpline(
    Y_AI,
    UP_AI,
    extrapolate=True
)


def Un_ai(x):
    return UN_AI_SPLINE(x)


def Up_ai(y):
    return UP_AI_SPLINE(y)


def Un_exp(x):
    return np.interp(
        x,
        X_EXP,
        UN_EXP
    )


def Up_exp(y):
    return np.interp(
        y,
        Y_EXP,
        UP_EXP
    )


# ------------------------------------------------------------
# 5. OCP 자체 비교
# ------------------------------------------------------------

fig, ax = plt.subplots(
    1,
    2,
    figsize=(11, 4.4)
)


ax[0].plot(
    X_AI,
    UN_AI,
    lw=2,
    label="Ai2020"
)

ax[0].plot(
    X_EXP,
    UN_EXP,
    lw=2,
    label="Experimental"
)

ax[0].set_title("Anode OCP")
ax[0].set_xlabel("Graphite stoichiometry x")
ax[0].set_ylabel("Voltage [V vs. Li/Li+]")
ax[0].legend()


ax[1].plot(
    Y_AI,
    UP_AI,
    lw=2,
    label="Ai2020"
)

ax[1].plot(
    Y_EXP,
    UP_EXP,
    lw=2,
    label="Experimental"
)

ax[1].set_title("Cathode OCP")
ax[1].set_xlabel("LCO stoichiometry y")
ax[1].set_ylabel("Voltage [V vs. Li/Li+]")
ax[1].legend()


plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Experimental OCP → PyBaMM parameter에 실제 주입
# ============================================================


def pybamm_interp(grid, values, name):

    grid = np.asarray(
        grid,
        dtype=float
    )

    values = np.asarray(
        values,
        dtype=float
    )

    def ocp(sto):

        return pybamm.Interpolant(
            grid,
            values,
            [sto],
            name=name,
            interpolator="linear",
            extrapolate=True,
        )

    return ocp


# Stage 0 correction이 적용된 Ai2020 복사
OCP_REPLACED_PARAMS = AI2020_CORRECTED.copy()


# OCP 두 개만 Experimental로 교체
OCP_REPLACED_PARAMS.update({

    "Negative electrode OCP [V]":
        pybamm_interp(
            X_EXP,
            UN_EXP,
            "Experimental anode OCP",
        ),

    "Positive electrode OCP [V]":
        pybamm_interp(
            Y_EXP,
            UP_EXP,
            "Experimental cathode OCP",
        ),
})


print("=== OCP replacement ===")

print(
    "Anode OCP replaced:",
    OCP_REPLACED_PARAMS[
        "Negative electrode OCP [V]"
    ]
    is not
    AI2020_CORRECTED[
        "Negative electrode OCP [V]"
    ]
)

print(
    "Cathode OCP replaced:",
    OCP_REPLACED_PARAMS[
        "Positive electrode OCP [V]"
    ]
    is not
    AI2020_CORRECTED[
        "Positive electrode OCP [V]"
    ]
)

print(
    "Experimental anode coverage:",
    X_EXP.min(),
    "to",
    X_EXP.max()
)

print(
    "Experimental cathode coverage:",
    Y_EXP.min(),
    "to",
    Y_EXP.max()
)

In [ ]:
# ============================================================
# OCP 교체 전 / 후 차이 확인
# ============================================================


# Experimental data가 존재하는 공통 영역에서만 비교
X_COMPARE = np.linspace(
    max(X_AI.min(), X_EXP.min()),
    min(X_AI.max(), X_EXP.max()),
    500
)

Y_COMPARE = np.linspace(
    max(Y_AI.min(), Y_EXP.min()),
    min(Y_AI.max(), Y_EXP.max()),
    500
)


UN_BEFORE = Un_ai(X_COMPARE)
UN_AFTER = Un_exp(X_COMPARE)

UP_BEFORE = Up_ai(Y_COMPARE)
UP_AFTER = Up_exp(Y_COMPARE)


DELTA_UN_MV = (
    UN_AFTER - UN_BEFORE
) * 1000

DELTA_UP_MV = (
    UP_AFTER - UP_BEFORE
) * 1000


fig, ax = plt.subplots(
    2,
    2,
    figsize=(11, 7)
)


# Anode OCP
ax[0, 0].plot(
    X_COMPARE,
    UN_BEFORE,
    label="Before: Ai2020"
)

ax[0, 0].plot(
    X_COMPARE,
    UN_AFTER,
    label="After: Experimental"
)

ax[0, 0].set_title("Anode OCP replacement")
ax[0, 0].set_xlabel("Graphite stoichiometry x")
ax[0, 0].set_ylabel("Voltage [V]")
ax[0, 0].legend()


# Cathode OCP
ax[0, 1].plot(
    Y_COMPARE,
    UP_BEFORE,
    label="Before: Ai2020"
)

ax[0, 1].plot(
    Y_COMPARE,
    UP_AFTER,
    label="After: Experimental"
)

ax[0, 1].set_title("Cathode OCP replacement")
ax[0, 1].set_xlabel("LCO stoichiometry y")
ax[0, 1].set_ylabel("Voltage [V]")
ax[0, 1].legend()


# Anode difference
ax[1, 0].plot(
    X_COMPARE,
    DELTA_UN_MV
)

ax[1, 0].axhline(
    0,
    linewidth=1
)

ax[1, 0].set_title(
    "Anode ΔOCP"
)

ax[1, 0].set_xlabel(
    "Graphite stoichiometry x"
)

ax[1, 0].set_ylabel(
    "Experimental - Ai2020 [mV]"
)


# Cathode difference
ax[1, 1].plot(
    Y_COMPARE,
    DELTA_UP_MV
)

ax[1, 1].axhline(
    0,
    linewidth=1
)

ax[1, 1].set_title(
    "Cathode ΔOCP"
)

ax[1, 1].set_xlabel(
    "LCO stoichiometry y"
)

ax[1, 1].set_ylabel(
    "Experimental - Ai2020 [mV]"
)


plt.tight_layout()
plt.show()


print(
    "Anode mean |ΔOCP| =",
    np.mean(np.abs(DELTA_UN_MV)),
    "mV"
)

print(
    "Anode max |ΔOCP|  =",
    np.max(np.abs(DELTA_UN_MV)),
    "mV"
)

print(
    "Cathode mean |ΔOCP| =",
    np.mean(np.abs(DELTA_UP_MV)),
    "mV"
)

print(
    "Cathode max |ΔOCP|  =",
    np.max(np.abs(DELTA_UP_MV)),
    "mV"
)

In [ ]:
# ============================================================
# Full-cell C/20 qOCV + electrode capacity
# ============================================================

SOC = np.linspace(0, 1, 1001)


def interp_unique(x, y, xnew):

    order = np.argsort(x)

    x = np.asarray(x)[order]
    y = np.asarray(y)[order]

    unique_x, index = np.unique(
        x,
        return_index=True
    )

    return np.interp(
        xnew,
        unique_x,
        y[index]
    )


# ------------------------------------------------------------
# 1. Full-cell C/20 CHG / DCHG
#
# 원본:
# 260808 Enertech셀 초기 저율 데이터 v2.xlsx
# ------------------------------------------------------------

capacity_chg, voltage_chg = FULLCELL_V2_CHG_BRANCH
capacity_dchg, voltage_dchg = FULLCELL_V2_DCHG_BRANCH


# Charge / discharge를 동일한 SOC 축으로 변환
V_CHG = interp_unique(
    capacity_chg / capacity_chg.max(),
    voltage_chg,
    SOC
)

V_DCH = interp_unique(
    1 - capacity_dchg / capacity_dchg.max(),
    voltage_dchg,
    SOC
)


# CHG / DCHG 평균 → full-cell quasi-OCV
V_QOCV = 0.5 * (
    V_CHG + V_DCH
)


# 측정 full-cell capacity
Q_MEAS = 0.5 * (
    capacity_chg.max()
    + capacity_dchg.max()
)


print(
    f"Bundle Q_MEAS={Q_MEAS_BUNDLE:.6f} Ah | "
    f"recomputed={Q_MEAS:.6f} Ah | "
    f"Δ={(Q_MEAS-Q_MEAS_BUNDLE)*1000:.3f} mAh"
)


# ------------------------------------------------------------
# 2. Ai2020 geometry 기반 electrode theoretical capacity
# ------------------------------------------------------------

CELL_AREA = (
    float(
        AI2020_CORRECTED[
            "Electrode height [m]"
        ]
    )
    *
    float(
        AI2020_CORRECTED[
            "Electrode width [m]"
        ]
    )
    *
    float(
        AI2020_CORRECTED[
            "Number of electrodes connected in parallel to make a cell"
        ]
    )
)


QN_CELL = (
    FARADAY_CONSTANT
    * CSN_MAX
    * EPS_N
    * LN
    * CELL_AREA
    / 3600
)


QP_CELL = (
    FARADAY_CONSTANT
    * CSP_MAX
    * EPS_P
    * LP
    * CELL_AREA
    / 3600
)


# ------------------------------------------------------------
# 3. Full-cell measured capacity에 해당하는 stoichiometry span
# ------------------------------------------------------------

DX = Q_MEAS / QN_CELL
DY = Q_MEAS / QP_CELL


print(f"Q_MEAS = {Q_MEAS:.6f} Ah")

print(
    f"Qn theoretical = {QN_CELL:.6f} Ah, "
    f"Δx = {DX:.6f}"
)

print(
    f"Qp theoretical = {QP_CELL:.6f} Ah, "
    f"Δy = {DY:.6f}"
)


assert 0 < DX < 1
assert 0 < DY < 1

# Stage 1 — Experimental OCP + 1A endpoint-constrained stoichiometry

Stage 1 방법론은 별도 `1A–1D validation notebook`에서 검토했고, 메인 모델에서는 **1A Endpoint-constrained balancing**을 사용한다.

고정되는 span:

\[
\Delta x = \frac{Q_{cell}}{Q_n}, \qquad
\Delta y = \frac{Q_{cell}}{Q_p}
\]

\[
x_{100}=x_0+\Delta x,\qquad
y_0=y_{100}+\Delta y
\]

`x0`, `y100`은 전체 qOCV RMSE를 최소화해서 찾지 않고, full-cell qOCV endpoint 조건을 만족하도록 결정한다.

\[
U_p(y_0)-U_n(x_0)=V_0
\]

\[
U_p(y_{100})-U_n(x_{100})=V_{100}
\]

RMSE / MAE는 결정된 window의 **검증 지표**로 계산한다.


In [ ]:
# ============================================================
# Stage 1 — 1A endpoint-constrained stoichiometry
# ============================================================

V0_TARGET = float(V_QOCV[0])
V100_TARGET = float(V_QOCV[-1])


def make_window_voltage(
    x0,
    x100,
    y100,
    y0,
):
    x = x0 + SOC * (x100 - x0)
    y = y0 - SOC * (y0 - y100)

    return (
        Up_exp(y)
        - Un_exp(x)
    )


def window_metrics(
    voltage,
):
    error_mV = (
        voltage
        - V_QOCV
    ) * 1000

    mid_mask = (
        (SOC >= 0.02)
        & (SOC <= 0.98)
    )

    return {
        "RMSE_full_mV":
            np.sqrt(
                np.mean(error_mV**2)
            ),

        "MAE_full_mV":
            np.mean(
                np.abs(error_mV)
            ),

        "RMSE_2_98_mV":
            np.sqrt(
                np.mean(
                    error_mV[mid_mask]**2
                )
            ),

        "MAE_2_98_mV":
            np.mean(
                np.abs(
                    error_mV[mid_mask]
                )
            ),
    }


def solve_stage1_endpoint():

    x0_lower = max(
        1e-6,
        float(X_EXP.min()),
    )

    x0_upper = min(
        1.0 - DX - 1e-6,
        float(X_EXP.max()) - DX,
    )

    y100_lower = max(
        1e-6,
        float(Y_EXP.min()),
    )

    y100_upper = min(
        1.0 - DY - 1e-6,
        float(Y_EXP.max()) - DY,
    )

    if (
        x0_upper <= x0_lower
        or y100_upper <= y100_lower
    ):
        raise RuntimeError(
            "Experimental OCP coverage가 "
            "필요 stoichiometry span을 덮지 못합니다."
        )

    def residuals(offsets):

        x0, y100 = offsets

        x100 = (
            x0 + DX
        )

        y0 = (
            y100 + DY
        )

        voltage_0 = (
            Up_exp(y0)
            - Un_exp(x0)
        )

        voltage_100 = (
            Up_exp(y100)
            - Un_exp(x100)
        )

        return np.array([
            voltage_0 - V0_TARGET,
            voltage_100 - V100_TARGET,
        ])

    initial_guess = np.array([
        0.5 * (
            x0_lower
            + x0_upper
        ),
        0.5 * (
            y100_lower
            + y100_upper
        ),
    ])

    result = least_squares(
        residuals,
        x0=initial_guess,
        bounds=(
            [
                x0_lower,
                y100_lower,
            ],
            [
                x0_upper,
                y100_upper,
            ],
        ),
        xtol=1e-12,
        ftol=1e-12,
        gtol=1e-12,
    )

    x0, y100 = result.x

    x100 = (
        x0 + DX
    )

    y0 = (
        y100 + DY
    )

    voltage = make_window_voltage(
        x0,
        x100,
        y100,
        y0,
    )

    metrics = window_metrics(
        voltage
    )

    endpoint_error_mV = (
        residuals(result.x)
        * 1000
    )

    return {
        "x0":
            float(x0),

        "x100":
            float(x100),

        "y100":
            float(y100),

        "y0":
            float(y0),

        "DX":
            float(DX),

        "DY":
            float(DY),

        "Endpoint_0_error_mV":
            float(
                endpoint_error_mV[0]
            ),

        "Endpoint_100_error_mV":
            float(
                endpoint_error_mV[1]
            ),

        **metrics,

        "Solver_success":
            bool(result.success),
    }


STAGE1 = solve_stage1_endpoint()


VOLTAGE_STAGE1 = make_window_voltage(
    STAGE1["x0"],
    STAGE1["x100"],
    STAGE1["y100"],
    STAGE1["y0"],
)


STAGE1_TABLE = pd.DataFrame([
    {
        "Method":
            "1A Endpoint-constrained",

        "Qn_Ah":
            QN_CELL,

        "Qp_Ah":
            QP_CELL,

        **STAGE1,
    }
])


display(
    STAGE1_TABLE.round(5)
)


plt.figure(
    figsize=(8, 5)
)

plt.plot(
    SOC,
    V_QOCV,
    lw=2.5,
    label="Measured C/20 qOCV",
)

plt.plot(
    SOC,
    VOLTAGE_STAGE1,
    lw=2,
    label="Stage 1 — 1A",
)

plt.xlabel("SOC")
plt.ylabel("Cell OCV [V]")
plt.title("Stage 1 qOCV validation")
plt.legend()
plt.grid()

plt.show()


## 공통 DFN parameter builder

Stage 1에서 확정한 Experimental OCP와 1A stoichiometry는 이후 Stage 2–4에서 고정한다.

추정 후보:
- `kn_mult`
- `kp_mult`
- `Dsn_mult`
- `Dsp_mult`
- `brugg_n`
- `brugg_p`
- `brugg_s`

Stage 2–3에서 OCP / stoichiometry는 다시 fitting하지 않는다.


In [ ]:
# ============================================================
# Common DFN parameter builder
# Stage 1 OCP + stoichiometry fixed
# ============================================================

BRUGG_N_AI2020 = float(
    AI2020_CORRECTED[
        "Negative electrode Bruggeman coefficient (electrolyte)"
    ]
)

BRUGG_P_AI2020 = float(
    AI2020_CORRECTED[
        "Positive electrode Bruggeman coefficient (electrolyte)"
    ]
)

BRUGG_S_AI2020 = float(
    AI2020_CORRECTED[
        "Separator Bruggeman coefficient (electrolyte)"
    ]
)


PARS_NOMINAL = {
    "kn_mult": 1.0,
    "kp_mult": 1.0,
    "Dsn_mult": 1.0,
    "Dsp_mult": 1.0,
    "brugg_n": BRUGG_N_AI2020,
    "brugg_p": BRUGG_P_AI2020,
    "brugg_s": BRUGG_S_AI2020,
}


PARS_BRUGG_15 = (
    PARS_NOMINAL.copy()
)

PARS_BRUGG_15.update({
    "brugg_n": 1.5,
    "brugg_p": 1.5,
    "brugg_s": 1.5,
})


PARAMS_7 = [
    "kn_mult",
    "kp_mult",
    "Dsn_mult",
    "Dsp_mult",
    "brugg_n",
    "brugg_p",
    "brugg_s",
]


PARAMS_4 = [
    "kn_mult",
    "kp_mult",
    "Dsn_mult",
    "Dsp_mult",
]


PARAM_LABELS = {
    "kn_mult": "k_n",
    "kp_mult": "k_p",
    "Dsn_mult": "D_s,n",
    "Dsp_mult": "D_s,p",
    "brugg_n": "b_n",
    "brugg_p": "b_p",
    "brugg_s": "b_s",
}


PARAM_BOUNDS = {
    "kn_mult": (0.20, 5.00),
    "kp_mult": (0.20, 5.00),
    "Dsn_mult": (0.10, 10.00),
    "Dsp_mult": (0.10, 10.00),
    "brugg_n": (1.00, 4.00),
    "brugg_p": (1.00, 4.00),
    "brugg_s": (1.00, 4.00),
}


def make_dfn_parameter_values(
    x_init,
    y_init,
    pars,
):

    base = (
        AI2020_CORRECTED.copy()
    )

    pv = base.copy()


    neg_j0_base = base[
        "Negative electrode exchange-current density [A.m-2]"
    ]

    pos_j0_base = base[
        "Positive electrode exchange-current density [A.m-2]"
    ]

    dsn_base = base[
        "Negative particle diffusivity [m2.s-1]"
    ]

    dsp_base = base[
        "Positive particle diffusivity [m2.s-1]"
    ]


    pv.update({

        "Initial concentration in negative electrode [mol.m-3]":
            float(x_init)
            * CSN_MAX,

        "Initial concentration in positive electrode [mol.m-3]":
            float(y_init)
            * CSP_MAX,


        "Negative electrode OCP [V]":
            pybamm_interp(
                X_EXP,
                UN_EXP,
                "Experimental negative OCP",
            ),

        "Positive electrode OCP [V]":
            pybamm_interp(
                Y_EXP,
                UP_EXP,
                "Experimental positive OCP",
            ),


        "Negative electrode exchange-current density [A.m-2]":
            scale_parameter_function(
                neg_j0_base,
                pars["kn_mult"],
            ),

        "Positive electrode exchange-current density [A.m-2]":
            scale_parameter_function(
                pos_j0_base,
                pars["kp_mult"],
            ),


        "Negative particle diffusivity [m2.s-1]":
            scale_parameter_function(
                dsn_base,
                pars["Dsn_mult"],
            ),

        "Positive particle diffusivity [m2.s-1]":
            scale_parameter_function(
                dsp_base,
                pars["Dsp_mult"],
            ),


        "Negative electrode Bruggeman coefficient (electrolyte)":
            float(
                pars["brugg_n"]
            ),

        "Positive electrode Bruggeman coefficient (electrolyte)":
            float(
                pars["brugg_p"]
            ),

        "Separator Bruggeman coefficient (electrolyte)":
            float(
                pars["brugg_s"]
            ),


        "Initial temperature [K]":
            T_REF,

        "Ambient temperature [K]":
            T_REF,

    }, check_already_exists=False)


    return pv


print(
    "Ai2020 Bruggeman:",
    BRUGG_N_AI2020,
    BRUGG_P_AI2020,
    BRUGG_S_AI2020,
)


## Stage 1 baseline — C-rate dynamic response

이 결과는 parameter fitting용이 아니다.  
Stage 1 OCP + stoichiometry만 적용했을 때의 **baseline**으로 저장한다.

Stage 4에서 Stage 3A / 3B fitting 결과와 동일한 0.5C / 1C / 2C Charge·Discharge 데이터로 다시 비교한다.


In [ ]:
# ============================================================
# Stage 1 baseline dynamic validation
# ============================================================

def load_dynamic_from_bundle():

    out = {}

    for rate in [
        0.5,
        1.0,
        2.0,
    ]:

        for charge in [
            True,
            False,
        ]:

            direction = (
                "charge"
                if charge
                else "discharge"
            )

            selected = DYNAMIC_DF[
                np.isclose(
                    pd.to_numeric(
                        DYNAMIC_DF["rate_C"],
                        errors="coerce",
                    ),
                    rate,
                )
                &
                (
                    DYNAMIC_DF["direction"]
                    .astype(str)
                    .str.lower()
                    == direction
                )
            ].copy()

            assert (
                len(selected) > 5
            ), (
                f"Dynamic data missing: "
                f"{rate}C {direction}"
            )

            out[
                (rate, charge)
            ] = {
                "t_min":
                    pd.to_numeric(
                        selected["t_min"],
                        errors="coerce",
                    ).to_numpy(
                        dtype=float
                    ),

                "V":
                    pd.to_numeric(
                        selected["V_V"],
                        errors="coerce",
                    ).to_numpy(
                        dtype=float
                    ),

                "SOC":
                    pd.to_numeric(
                        selected["SOC"],
                        errors="coerce",
                    ).to_numpy(
                        dtype=float
                    ),
            }

    return out


EXP_DYN = (
    load_dynamic_from_bundle()
)


def run_cc_dfn(
    rate,
    charge,
    pars,
):

    x_init = (
        STAGE1["x0"]
        if charge
        else STAGE1["x100"]
    )

    y_init = (
        STAGE1["y0"]
        if charge
        else STAGE1["y100"]
    )


    model = pybamm.lithium_ion.DFN({
        "thermal": "isothermal",
    })


    model.variables[
        "Anode potential [V]"
    ] = model.variables[
        "Negative electrode surface potential difference at separator interface [V]"
    ]


    step = (
        f"Charge at {rate}C until 4.2 V"
        if charge
        else
        f"Discharge at {rate}C until 3.0 V"
    )


    sim = pybamm.Simulation(

        model,

        parameter_values=
            make_dfn_parameter_values(
                x_init,
                y_init,
                pars,
            ),

        experiment=
            pybamm.Experiment(
                [step],
                period="10 seconds",
            ),

        var_pts={
            "x_n": 15,
            "x_s": 15,
            "x_p": 15,
            "r_n": 15,
            "r_p": 15,
        },

        solver=
            pybamm.IDAKLUSolver(
                rtol=1e-6,
                atol=1e-8,
            ),
    )


    sol = sim.solve()


    time_s = np.asarray(
        sol["Time [s]"].entries,
        dtype=float,
    )

    voltage = np.asarray(
        sol[
            "Terminal voltage [V]"
        ].entries,
        dtype=float,
    )

    current = np.asarray(
        sol[
            "Current [A]"
        ].entries,
        dtype=float,
    )

    anode_potential = np.asarray(
        sol[
            "Anode potential [V]"
        ].entries,
        dtype=float,
    ).squeeze()


    capacity_abs = (
        cumulative_trapezoid(
            np.abs(current),
            time_s,
            initial=0.0,
        )
        / 3600
    )


    soc = (
        capacity_abs / Q_MEAS
        if charge
        else
        1.0 - capacity_abs / Q_MEAS
    )


    return {
        "t_min":
            time_s / 60,

        "V":
            voltage,

        "SOC":
            np.clip(
                soc,
                0,
                1,
            ),

        "Anode_potential_V":
            anode_potential,
    }


STAGE1_BASELINE_SIMS = {}


for rate in [
    0.5,
    1.0,
    2.0,
]:

    for charge in [
        True,
        False,
    ]:

        print(
            "Stage 1 baseline:",
            rate,
            "C",
            (
                "Charge"
                if charge
                else "Discharge"
            ),
        )

        try:

            STAGE1_BASELINE_SIMS[
                (rate, charge)
            ] = run_cc_dfn(
                rate,
                charge,
                PARS_NOMINAL,
            )

        except Exception as error:

            print(
                "FAILED:",
                error
            )


# Stage 2 — BoL HPPC sensitivity / identifiability

## BoL HPPC calibration data

사용 파일: `260808 Eneterch셀 열화데이터 v1 6-6(5).xlsx`

최초 HPPC는 Cycle 1–10에서 반복된다.

각 SOC block:
1. Rest 10 min
2. 1C discharge pulse 30 s (`~2.28 A`)
3. Rest 40 s
4. 0.75C charge pulse 10 s (`~1.71 A`)
5. 다음 SOC 이동용 discharge (`~0.76 A`, Cycle 1–9에서 18 min)

### Fitting target
raw `record` 시트의 기록 간격을 고려해 각 block에서 5개 대표 전압점을 사용한다.

- pre-rest end
- discharge 30 s end
- rest 30 s
- rest 40 s end
- charge 10 s end

Stage 2–3에서 **0.5C / 1C / 2C CC data는 사용하지 않는다.**


In [ ]:
# ============================================================
# BoL HPPC raw data load / key-point extraction
# ============================================================

HPPC_FILE = (
    ROOT
    / "260808 Eneterch셀 열화데이터 v1 6-6(5).xlsx"
)

print(
    "HPPC raw file:",
    HPPC_FILE
)


HPPC_STEP_DF = pd.read_excel(
    HPPC_FILE,
    sheet_name="step",
)

HPPC_RECORD_DF = pd.read_excel(
    HPPC_FILE,
    sheet_name="record",
)


for col in [
    "Oneset Date",
    "End Date",
]:
    HPPC_STEP_DF[col] = pd.to_datetime(
        HPPC_STEP_DF[col]
    )


HPPC_RECORD_DF["Date"] = pd.to_datetime(
    HPPC_RECORD_DF["Date"]
)


HPPC_STEP_DF[
    "Cycle Index"
] = pd.to_numeric(
    HPPC_STEP_DF[
        "Cycle Index"
    ],
    errors="coerce",
)


HPPC_STEP_DF[
    "Step Index"
] = pd.to_numeric(
    HPPC_STEP_DF[
        "Step Index"
    ],
    errors="coerce",
)


def get_hppc_step(
    cycle,
    step_index,
):

    selected = HPPC_STEP_DF[
        (
            HPPC_STEP_DF[
                "Cycle Index"
            ]
            == cycle
        )
        &
        (
            HPPC_STEP_DF[
                "Step Index"
            ]
            == step_index
        )
    ]

    if len(selected) != 1:
        raise RuntimeError(
            f"HPPC step not unique: "
            f"cycle={cycle}, step={step_index}"
        )

    return selected.iloc[0]


def get_record_segment(
    step_row,
):

    onset = step_row[
        "Oneset Date"
    ]

    end = step_row[
        "End Date"
    ]

    step_type = str(
        step_row[
            "Step Type"
        ]
    )


    segment = HPPC_RECORD_DF[
        (
            HPPC_RECORD_DF["Date"]
            >= onset
        )
        &
        (
            HPPC_RECORD_DF["Date"]
            <= end
        )
        &
        (
            HPPC_RECORD_DF[
                "Step Type"
            ].astype(str)
            == step_type
        )
    ].copy()


    if segment.empty:
        raise RuntimeError(
            f"No record segment: "
            f"{step_type}, "
            f"{onset} ~ {end}"
        )


    segment = segment.sort_values(
        "Date"
    )


    segment["Voltage(V)"] = pd.to_numeric(
        segment["Voltage(V)"],
        errors="coerce",
    )


    return segment


def closest_voltage(
    segment,
    target_time,
):

    index = (
        segment["Date"]
        - target_time
    ).abs().idxmin()

    return float(
        segment.loc[
            index,
            "Voltage(V)"
        ]
    )


hppc_rows = []


for cycle in range(
    1,
    11
):

    rest_pre = get_hppc_step(
        cycle,
        24,
    )

    dchg = get_hppc_step(
        cycle,
        25,
    )

    rest = get_hppc_step(
        cycle,
        26,
    )

    chg = get_hppc_step(
        cycle,
        27,
    )


    rest_pre_data = (
        get_record_segment(
            rest_pre
        )
    )

    dchg_data = (
        get_record_segment(
            dchg
        )
    )

    rest_data = (
        get_record_segment(
            rest
        )
    )

    chg_data = (
        get_record_segment(
            chg
        )
    )


    nominal_soc = (
        1.0
        - 0.1
        * (cycle - 1)
    )


    values = {
        "rest_pre_end":
            float(
                rest_pre_data[
                    "Voltage(V)"
                ].iloc[-1]
            ),

        "dchg_end":
            float(
                dchg_data[
                    "Voltage(V)"
                ].iloc[-1]
            ),

        "rest_30s":
            closest_voltage(
                rest_data,
                rest["Oneset Date"]
                + pd.Timedelta(
                    seconds=30
                ),
            ),

        "rest_end":
            float(
                rest_data[
                    "Voltage(V)"
                ].iloc[-1]
            ),

        "chg_end":
            float(
                chg_data[
                    "Voltage(V)"
                ].iloc[-1]
            ),
    }


    for point_type, voltage in (
        values.items()
    ):

        hppc_rows.append({
            "Cycle":
                cycle,

            "Nominal_SOC":
                nominal_soc,

            "Point":
                point_type,

            "Voltage_V":
                voltage,
        })


HPPC_TARGETS = pd.DataFrame(
    hppc_rows
)


HPPC_POINT_ORDER = [
    "rest_pre_end",
    "dchg_end",
    "rest_30s",
    "rest_end",
    "chg_end",
]


HPPC_TARGETS[
    "Point"
] = pd.Categorical(
    HPPC_TARGETS["Point"],
    categories=HPPC_POINT_ORDER,
    ordered=True,
)


HPPC_TARGETS = (
    HPPC_TARGETS
    .sort_values(
        [
            "Cycle",
            "Point",
        ]
    )
    .reset_index(
        drop=True
    )
)


HPPC_EXP_VECTOR = (
    HPPC_TARGETS[
        "Voltage_V"
    ]
    .to_numpy(
        dtype=float
    )
)


display(
    HPPC_TARGETS
)


print(
    "HPPC target points =",
    len(
        HPPC_EXP_VECTOR
    )
)


In [ ]:
# ============================================================
# HPPC protocol simulation
# Entire BoL HPPC is simulated sequentially
# ============================================================

HPPC_DCHG_CURRENT_A = 2.28
HPPC_CHG_CURRENT_A = 1.71
HPPC_SHIFT_CURRENT_A = 0.76


def make_hppc_experiment():

    steps = []

    for cycle in range(
        1,
        11
    ):

        steps.extend([
            "Rest for 10 minutes",
            (
                f"Discharge at "
                f"{HPPC_DCHG_CURRENT_A} A "
                f"for 30 seconds"
            ),
            "Rest for 40 seconds",
            (
                f"Charge at "
                f"{HPPC_CHG_CURRENT_A} A "
                f"for 10 seconds"
            ),
        ])


        if cycle < 10:

            steps.append(
                (
                    f"Discharge at "
                    f"{HPPC_SHIFT_CURRENT_A} A "
                    f"for 18 minutes"
                )
            )


    return pybamm.Experiment(
        steps,
        period="10 seconds",
    )


HPPC_EXPERIMENT = (
    make_hppc_experiment()
)


def make_hppc_target_times():

    rows = []

    time_s = 0.0

    for cycle in range(
        1,
        11
    ):

        time_s += 600.0

        rows.append({
            "Cycle": cycle,
            "Point": "rest_pre_end",
            "Time_s": time_s,
        })


        time_s += 30.0

        rows.append({
            "Cycle": cycle,
            "Point": "dchg_end",
            "Time_s": time_s,
        })


        time_s += 30.0

        rows.append({
            "Cycle": cycle,
            "Point": "rest_30s",
            "Time_s": time_s,
        })


        time_s += 10.0

        rows.append({
            "Cycle": cycle,
            "Point": "rest_end",
            "Time_s": time_s,
        })


        time_s += 10.0

        rows.append({
            "Cycle": cycle,
            "Point": "chg_end",
            "Time_s": time_s,
        })


        if cycle < 10:
            time_s += 1080.0


    return pd.DataFrame(
        rows
    )


HPPC_TARGET_TIMES = (
    make_hppc_target_times()
)


def interp_first_duplicate(
    time,
    values,
    target_time,
):

    time = np.asarray(
        time,
        dtype=float,
    )

    values = np.asarray(
        values,
        dtype=float,
    )


    order = np.argsort(
        time,
        kind="stable",
    )

    time = time[order]
    values = values[order]


    unique_time, index = np.unique(
        time,
        return_index=True,
    )

    unique_values = values[
        index
    ]


    return np.interp(
        target_time,
        unique_time,
        unique_values,
    )


HPPC_SIM_CACHE = {}


def hppc_pars_key(
    pars,
):

    return tuple(
        round(
            float(
                pars[name]
            ),
            8,
        )
        for name in PARAMS_7
    )


def run_hppc_dfn(
    pars,
    return_solution=False,
):

    key = hppc_pars_key(
        pars
    )


    if (
        key in HPPC_SIM_CACHE
        and not return_solution
    ):
        return HPPC_SIM_CACHE[
            key
        ]


    model = pybamm.lithium_ion.DFN({
        "thermal": "isothermal",
    })


    simulation = pybamm.Simulation(

        model,

        parameter_values=
            make_dfn_parameter_values(
                STAGE1["x100"],
                STAGE1["y100"],
                pars,
            ),

        experiment=
            HPPC_EXPERIMENT,

        var_pts={
            "x_n": 15,
            "x_s": 15,
            "x_p": 15,
            "r_n": 15,
            "r_p": 15,
        },

        solver=
            pybamm.IDAKLUSolver(
                rtol=1e-6,
                atol=1e-8,
            ),
    )


    solution = (
        simulation.solve()
    )


    time_s = np.asarray(
        solution[
            "Time [s]"
        ].entries,
        dtype=float,
    )

    voltage = np.asarray(
        solution[
            "Terminal voltage [V]"
        ].entries,
        dtype=float,
    )


    model_vector = np.array([
        interp_first_duplicate(
            time_s,
            voltage,
            target_time,
        )
        for target_time in (
            HPPC_TARGET_TIMES[
                "Time_s"
            ]
        )
    ])


    if not return_solution:

        HPPC_SIM_CACHE[
            key
        ] = model_vector

        return model_vector


    return (
        model_vector,
        solution,
    )


HPPC_NOMINAL_VECTOR = run_hppc_dfn(
    PARS_NOMINAL
)


HPPC_NOMINAL_ERROR_MV = (
    HPPC_NOMINAL_VECTOR
    - HPPC_EXP_VECTOR
) * 1000


print(
    "Stage 1 HPPC baseline RMSE =",
    np.sqrt(
        np.mean(
            HPPC_NOMINAL_ERROR_MV**2
        )
    ),
    "mV",
)

print(
    "Stage 1 HPPC baseline MAE =",
    np.mean(
        np.abs(
            HPPC_NOMINAL_ERROR_MV
        )
    ),
    "mV",
)


## Stage 2A — 7-parameter HPPC sensitivity / identifiability

Baseline:
- Stage 1 Experimental OCP + 1A stoichiometry 고정
- Ai2020 nominal kinetics / diffusion / Bruggeman

후보:

\[
\{k_n,k_p,D_{s,n},D_{s,p},b_n,b_p,b_s\}
\]

Main perturbation은 5%를 사용한다.  
Bruggeman은 exponent 자체의 단순 ±5%가 아니라, `ε^b` effective transport가 동일 log-step만큼 변하도록 perturbation한다.

출력:
- HPPC key-point voltage sensitivity
- RMS sensitivity
- cosine similarity
- singular values
- condition number / effective rank
- auto-screened subset


In [ ]:
# ============================================================
# HPPC sensitivity / identifiability helpers
# ============================================================

SENS_LOG_STEP = np.log(
    1.05
)


BRUGG_POROSITY = {
    "brugg_n":
        float(
            AI2020_CORRECTED[
                "Negative electrode porosity"
            ]
        ),

    "brugg_p":
        float(
            AI2020_CORRECTED[
                "Positive electrode porosity"
            ]
        ),

    "brugg_s":
        float(
            AI2020_CORRECTED[
                "Separator porosity"
            ]
        ),
}


def perturb_for_lsa(
    base,
    parameter,
    direction,
):

    pars = base.copy()


    if parameter.startswith(
        "brugg_"
    ):

        porosity = (
            BRUGG_POROSITY[
                parameter
            ]
        )

        delta_b = (
            direction
            * SENS_LOG_STEP
            / np.log(
                porosity
            )
        )

        pars[parameter] = (
            float(
                base[parameter]
            )
            + delta_b
        )

    else:

        pars[parameter] = (
            float(
                base[parameter]
            )
            * np.exp(
                direction
                * SENS_LOG_STEP
            )
        )


    return pars


def compute_hppc_sensitivity(
    base,
    params,
    stage_name,
):

    columns = []
    rows = []


    for parameter in params:

        print(
            stage_name,
            parameter
        )


        pars_plus = perturb_for_lsa(
            base,
            parameter,
            +1,
        )

        pars_minus = perturb_for_lsa(
            base,
            parameter,
            -1,
        )


        voltage_plus = run_hppc_dfn(
            pars_plus
        )

        voltage_minus = run_hppc_dfn(
            pars_minus
        )


        sensitivity = (
            voltage_plus
            - voltage_minus
        ) / (
            2.0
            * SENS_LOG_STEP
        )


        columns.append(
            sensitivity
        )


        rows.append({
            "Stage":
                stage_name,

            "Parameter":
                parameter,

            "Label":
                PARAM_LABELS[
                    parameter
                ],

            "RMS_mV_per_log_effect":
                np.sqrt(
                    np.mean(
                        sensitivity**2
                    )
                )
                * 1000,

            "MaxAbs_mV_per_log_effect":
                np.max(
                    np.abs(
                        sensitivity
                    )
                )
                * 1000,
        })


    matrix = np.column_stack(
        columns
    )


    return (
        matrix,
        pd.DataFrame(
            rows
        ),
    )


def identifiability_report(
    matrix,
    params,
):

    column_norm = np.linalg.norm(
        matrix,
        axis=0,
    )


    safe_norm = np.where(
        column_norm > 0,
        column_norm,
        1.0,
    )


    normalized = (
        matrix
        / safe_norm
    )


    cosine = (
        normalized.T
        @ normalized
    )


    singular_values = (
        np.linalg.svd(
            normalized,
            compute_uv=False,
        )
    )


    relative_sigma = (
        singular_values
        / singular_values[0]
    )


    condition_number = (
        np.inf
        if singular_values[-1] == 0
        else
        singular_values[0]
        / singular_values[-1]
    )


    singular_table = pd.DataFrame({
        "Index":
            np.arange(
                1,
                len(
                    singular_values
                ) + 1
            ),

        "Singular_value":
            singular_values,

        "Relative_sigma":
            relative_sigma,
    })


    return {
        "cosine":
            pd.DataFrame(
                cosine,
                index=params,
                columns=params,
            ),

        "singular_values":
            singular_table,

        "condition_number":
            condition_number,

        "rank_rel_1e2":
            int(
                np.sum(
                    relative_sigma
                    >= 1e-2
                )
            ),

        "rank_rel_1e3":
            int(
                np.sum(
                    relative_sigma
                    >= 1e-3
                )
            ),
    }


def greedy_subset(
    matrix,
    params,
    relative_sensitivity_floor=0.10,
    cosine_threshold=0.95,
):

    rms = np.sqrt(
        np.mean(
            matrix**2,
            axis=0,
        )
    )

    rms_rel = (
        rms
        / np.max(rms)
    )


    column_norm = np.linalg.norm(
        matrix,
        axis=0,
    )


    normalized = (
        matrix
        / np.where(
            column_norm > 0,
            column_norm,
            1.0,
        )
    )


    cosine = (
        normalized.T
        @ normalized
    )


    order = np.argsort(
        -rms
    )


    selected = []


    for index in order:

        if (
            rms_rel[index]
            < relative_sensitivity_floor
        ):
            continue


        if not selected:

            selected.append(
                index
            )

            continue


        max_collinearity = max(
            abs(
                cosine[
                    index,
                    selected_index
                ]
            )
            for selected_index
            in selected
        )


        if (
            max_collinearity
            < cosine_threshold
        ):

            selected.append(
                index
            )


    return [
        params[index]
        for index in selected
    ]


In [ ]:
# ============================================================
# Stage 2A — 7 parameters
# ============================================================

SENS_BASE_7 = (
    PARS_NOMINAL.copy()
)


S7, SENSITIVITY_7 = (
    compute_hppc_sensitivity(
        SENS_BASE_7,
        PARAMS_7,
        "Stage 2A",
    )
)


ID7 = identifiability_report(
    S7,
    PARAMS_7,
)


AUTO_SUBSET_7 = greedy_subset(
    S7,
    PARAMS_7,
)


display(
    SENSITIVITY_7
    .sort_values(
        "RMS_mV_per_log_effect",
        ascending=False,
    )
    .round(3)
)


print(
    "Condition number =",
    ID7[
        "condition_number"
    ]
)

print(
    "Rank (relative sigma >= 1e-2) =",
    ID7[
        "rank_rel_1e2"
    ]
)

print(
    "Rank (relative sigma >= 1e-3) =",
    ID7[
        "rank_rel_1e3"
    ]
)


display(
    ID7[
        "cosine"
    ].round(3)
)


display(
    ID7[
        "singular_values"
    ].round(5)
)


print(
    "Stage 2A auto subset =",
    AUTO_SUBSET_7
)


## Stage 2B — Bruggeman = 1.5 고정 후 4-parameter 재분석

실제 baseline parameter set을:

\[
b_n=b_p=b_s=1.5
\]

로 바꾼 뒤 아래 4개에 대해 HPPC sensitivity / identifiability를 **처음부터 다시 계산**한다.

\[
\{k_n,k_p,D_{s,n},D_{s,p}\}
\]


In [ ]:
# ============================================================
# Stage 2B — Bruggeman = 1.5 fixed
# ============================================================

SENS_BASE_4 = (
    PARS_BRUGG_15.copy()
)


S4, SENSITIVITY_4 = (
    compute_hppc_sensitivity(
        SENS_BASE_4,
        PARAMS_4,
        "Stage 2B",
    )
)


ID4 = identifiability_report(
    S4,
    PARAMS_4,
)


AUTO_SUBSET_4 = greedy_subset(
    S4,
    PARAMS_4,
)


display(
    SENSITIVITY_4
    .sort_values(
        "RMS_mV_per_log_effect",
        ascending=False,
    )
    .round(3)
)


print(
    "Condition number =",
    ID4[
        "condition_number"
    ]
)

print(
    "Rank (relative sigma >= 1e-2) =",
    ID4[
        "rank_rel_1e2"
    ]
)

print(
    "Rank (relative sigma >= 1e-3) =",
    ID4[
        "rank_rel_1e3"
    ]
)


display(
    ID4[
        "cosine"
    ].round(3)
)


display(
    ID4[
        "singular_values"
    ].round(5)
)


print(
    "Stage 2B auto subset =",
    AUTO_SUBSET_4
)


# Stage 3A / 3B — BoL HPPC parameter fitting

## Calibration dataset
Stage 3에서는 **BoL HPPC 50 key-points만 사용**한다.  
0.5C / 1C / 2C CC data는 Stage 4 validation까지 보존한다.

## Objective

각 point type과 SOC block이 동일한 비중을 갖도록 residual을 정규화한다.

\[
J =
\frac{1}{2}
\sum_i
w_i
\left(
V_{model,i}-V_{exp,i}
\right)^2
\]

현재 모든 point type weight는 동일하게 1.0으로 시작한다.

## Optimization strategy

두 Stage 모두 동일한 computational strategy를 사용한다.

- Multi-start `least_squares` TRF
- Differential Evolution
- Dual Annealing
- DE / DA global best → TRF local refinement

결과에는 아래 항목을 모두 기록한다.

- Algorithm
- Start / seed
- Objective
- Function evaluations
- RMSE / MAE
- Optimized parameter
- Bound hit


In [ ]:
# ============================================================
# Stage 3 optimization configuration
#
# "quick" : code debugging
# "long"  : feasibility main run
# ============================================================

OPT_MODE = "long"


if OPT_MODE == "quick":

    TRF_STARTS = 3
    TRF_MAX_NFEV = 30

    DE_SEEDS = [
        11
    ]

    DE_MAXITER = 5
    DE_POPSIZE = 4

    DA_SEEDS = [
        11
    ]

    DA_MAXFUN = 100

    REFINE_MAX_NFEV = 40


else:

    TRF_STARTS = 10
    TRF_MAX_NFEV = 100

    DE_SEEDS = [
        11,
        22,
        33,
    ]

    DE_MAXITER = 15
    DE_POPSIZE = 5

    DA_SEEDS = [
        11,
        22,
        33,
    ]

    DA_MAXFUN = 500

    REFINE_MAX_NFEV = 100


USE_AUTO_SUBSET = True


OPT_BASE_3A = (
    PARS_NOMINAL.copy()
)

OPT_PARAMS_3A = (
    AUTO_SUBSET_7.copy()
    if (
        USE_AUTO_SUBSET
        and len(
            AUTO_SUBSET_7
        ) > 0
    )
    else PARAMS_7.copy()
)


OPT_BASE_3B = (
    PARS_BRUGG_15.copy()
)

OPT_PARAMS_3B = (
    AUTO_SUBSET_4.copy()
    if (
        USE_AUTO_SUBSET
        and len(
            AUTO_SUBSET_4
        ) > 0
    )
    else PARAMS_4.copy()
)


HPPC_POINT_WEIGHTS = {
    "rest_pre_end": 1.0,
    "dchg_end": 1.0,
    "rest_30s": 1.0,
    "rest_end": 1.0,
    "chg_end": 1.0,
}


print(
    "Stage 3A parameters =",
    OPT_PARAMS_3A
)

print(
    "Stage 3B parameters =",
    OPT_PARAMS_3B
)

print(
    "Optimization mode =",
    OPT_MODE
)


In [ ]:
# ============================================================
# Stage 3 objective / parameter encoding
# ============================================================

POINT_WEIGHT_VECTOR = np.array([
    HPPC_POINT_WEIGHTS[
        str(point)
    ]
    for point in (
        HPPC_TARGETS[
            "Point"
        ].astype(str)
    )
])


POINT_COUNT = {
    point:
        int(
            np.sum(
                HPPC_TARGETS[
                    "Point"
                ].astype(str)
                == point
            )
        )
    for point in HPPC_POINT_ORDER
}


NORMALIZED_WEIGHT_VECTOR = np.array([
    np.sqrt(
        HPPC_POINT_WEIGHTS[
            str(point)
        ]
        /
        POINT_COUNT[
            str(point)
        ]
    )
    for point in (
        HPPC_TARGETS[
            "Point"
        ].astype(str)
    )
])


def encode_params(
    pars,
    opt_params,
):

    return np.array([
        np.log(
            float(
                pars[parameter]
            )
        )
        for parameter
        in opt_params
    ])


def decode_params(
    z,
    base,
    opt_params,
):

    pars = base.copy()

    for parameter, value in zip(
        opt_params,
        z,
    ):

        pars[parameter] = float(
            np.exp(value)
        )

    return pars


def log_bounds(
    opt_params,
):

    lower = []
    upper = []

    for parameter in opt_params:

        lo, hi = PARAM_BOUNDS[
            parameter
        ]

        lower.append(
            np.log(lo)
        )

        upper.append(
            np.log(hi)
        )

    return (
        np.asarray(lower),
        np.asarray(upper),
    )


def hppc_residual_vector_from_pars(
    pars,
):

    model_vector = run_hppc_dfn(
        pars
    )

    error = (
        model_vector
        - HPPC_EXP_VECTOR
    )

    return (
        error
        * NORMALIZED_WEIGHT_VECTOR
    )


def evaluate_hppc_pars(
    pars,
):

    model_vector = run_hppc_dfn(
        pars
    )

    error_mV = (
        model_vector
        - HPPC_EXP_VECTOR
    ) * 1000


    rows = []

    for point in HPPC_POINT_ORDER:

        mask = (
            HPPC_TARGETS[
                "Point"
            ].astype(str)
            == point
        ).to_numpy()

        point_error = (
            error_mV[
                mask
            ]
        )

        rows.append({
            "Point":
                point,

            "RMSE_mV":
                np.sqrt(
                    np.mean(
                        point_error**2
                    )
                ),

            "MAE_mV":
                np.mean(
                    np.abs(
                        point_error
                    )
                ),
        })


    return {
        "RMSE_mV":
            np.sqrt(
                np.mean(
                    error_mV**2
                )
            ),

        "MAE_mV":
            np.mean(
                np.abs(
                    error_mV
                )
            ),

        "Point_table":
            pd.DataFrame(
                rows
            ),

        "Model_vector":
            model_vector,
    }


def bound_hit_string(
    pars,
    opt_params,
    tolerance=0.01,
):

    hits = []

    for parameter in opt_params:

        lo, hi = PARAM_BOUNDS[
            parameter
        ]

        value = float(
            pars[
                parameter
            ]
        )

        span = hi - lo

        if (
            abs(
                value - lo
            )
            <= tolerance * span
        ):
            hits.append(
                parameter
                + ":lower"
            )

        elif (
            abs(
                value - hi
            )
            <= tolerance * span
        ):
            hits.append(
                parameter
                + ":upper"
            )


    return (
        ", ".join(hits)
        if hits
        else ""
    )


In [ ]:
# ============================================================
# Stage 3 optimizer suite
# ============================================================

def run_optimizer_suite(
    case_name,
    base,
    opt_params,
):

    lower, upper = log_bounds(
        opt_params
    )

    z_nominal = np.clip(
        encode_params(
            base,
            opt_params,
        ),
        lower,
        upper,
    )


    result_rows = []
    candidates = []


    def residual_z(z):

        pars = decode_params(
            z,
            base,
            opt_params,
        )

        try:

            return (
                hppc_residual_vector_from_pars(
                    pars
                )
            )

        except Exception:

            return np.full(
                len(
                    HPPC_EXP_VECTOR
                ),
                0.20
            )


    def scalar_z(z):

        residual = residual_z(
            z
        )

        return 0.5 * float(
            np.sum(
                residual**2
            )
        )


    # --------------------------------------------------------
    # 1. TRF multi-start
    # --------------------------------------------------------

    rng = np.random.default_rng(
        42
    )


    starts = [
        z_nominal
    ]


    for _ in range(
        TRF_STARTS - 1
    ):

        starts.append(
            rng.uniform(
                lower,
                upper,
            )
        )


    for start_number, start in enumerate(
        starts,
        start=1,
    ):

        print(
            f"{case_name} | "
            f"TRF start "
            f"{start_number}/{TRF_STARTS}"
        )


        result = least_squares(
            residual_z,
            x0=start,
            bounds=(
                lower,
                upper,
            ),
            method="trf",
            max_nfev=TRF_MAX_NFEV,
            xtol=1e-5,
            ftol=1e-5,
            gtol=1e-5,
        )


        pars = decode_params(
            result.x,
            base,
            opt_params,
        )

        metrics = evaluate_hppc_pars(
            pars
        )


        candidates.append({
            "Algorithm":
                "TRF",

            "Run":
                start_number,

            "Seed":
                np.nan,

            "pars":
                pars,

            "RMSE_mV":
                metrics[
                    "RMSE_mV"
                ],

            "MAE_mV":
                metrics[
                    "MAE_mV"
                ],

            "Objective":
                scalar_z(
                    result.x
                ),

            "nfev":
                result.nfev,

            "Success":
                result.success,
        })


    # --------------------------------------------------------
    # 2. Differential Evolution + TRF refine
    # --------------------------------------------------------

    bounds_list = list(
        zip(
            lower,
            upper,
        )
    )


    for seed in DE_SEEDS:

        print(
            f"{case_name} | "
            f"DE seed={seed}"
        )


        result_de = differential_evolution(
            scalar_z,
            bounds=bounds_list,
            seed=seed,
            maxiter=DE_MAXITER,
            popsize=DE_POPSIZE,
            polish=False,
            updating="immediate",
        )


        pars_de = decode_params(
            result_de.x,
            base,
            opt_params,
        )

        metrics_de = (
            evaluate_hppc_pars(
                pars_de
            )
        )


        candidates.append({
            "Algorithm":
                "DE",

            "Run":
                np.nan,

            "Seed":
                seed,

            "pars":
                pars_de,

            "RMSE_mV":
                metrics_de[
                    "RMSE_mV"
                ],

            "MAE_mV":
                metrics_de[
                    "MAE_mV"
                ],

            "Objective":
                result_de.fun,

            "nfev":
                result_de.nfev,

            "Success":
                result_de.success,
        })


        print(
            f"{case_name} | "
            f"DE→TRF seed={seed}"
        )


        result_refine = least_squares(
            residual_z,
            x0=result_de.x,
            bounds=(
                lower,
                upper,
            ),
            method="trf",
            max_nfev=REFINE_MAX_NFEV,
            xtol=1e-6,
            ftol=1e-6,
            gtol=1e-6,
        )


        pars_refine = decode_params(
            result_refine.x,
            base,
            opt_params,
        )

        metrics_refine = (
            evaluate_hppc_pars(
                pars_refine
            )
        )


        candidates.append({
            "Algorithm":
                "DE→TRF",

            "Run":
                np.nan,

            "Seed":
                seed,

            "pars":
                pars_refine,

            "RMSE_mV":
                metrics_refine[
                    "RMSE_mV"
                ],

            "MAE_mV":
                metrics_refine[
                    "MAE_mV"
                ],

            "Objective":
                scalar_z(
                    result_refine.x
                ),

            "nfev":
                result_de.nfev
                + result_refine.nfev,

            "Success":
                result_refine.success,
        })


    # --------------------------------------------------------
    # 3. Dual Annealing + TRF refine
    # --------------------------------------------------------

    for seed in DA_SEEDS:

        print(
            f"{case_name} | "
            f"DA seed={seed}"
        )


        result_da = dual_annealing(
            scalar_z,
            bounds=bounds_list,
            seed=seed,
            maxfun=DA_MAXFUN,
            no_local_search=True,
        )


        pars_da = decode_params(
            result_da.x,
            base,
            opt_params,
        )

        metrics_da = (
            evaluate_hppc_pars(
                pars_da
            )
        )


        candidates.append({
            "Algorithm":
                "Dual Annealing",

            "Run":
                np.nan,

            "Seed":
                seed,

            "pars":
                pars_da,

            "RMSE_mV":
                metrics_da[
                    "RMSE_mV"
                ],

            "MAE_mV":
                metrics_da[
                    "MAE_mV"
                ],

            "Objective":
                result_da.fun,

            "nfev":
                result_da.nfev,

            "Success":
                bool(
                    result_da.success
                ),
        })


        print(
            f"{case_name} | "
            f"DA→TRF seed={seed}"
        )


        result_refine = least_squares(
            residual_z,
            x0=result_da.x,
            bounds=(
                lower,
                upper,
            ),
            method="trf",
            max_nfev=REFINE_MAX_NFEV,
            xtol=1e-6,
            ftol=1e-6,
            gtol=1e-6,
        )


        pars_refine = decode_params(
            result_refine.x,
            base,
            opt_params,
        )

        metrics_refine = (
            evaluate_hppc_pars(
                pars_refine
            )
        )


        candidates.append({
            "Algorithm":
                "DA→TRF",

            "Run":
                np.nan,

            "Seed":
                seed,

            "pars":
                pars_refine,

            "RMSE_mV":
                metrics_refine[
                    "RMSE_mV"
                ],

            "MAE_mV":
                metrics_refine[
                    "MAE_mV"
                ],

            "Objective":
                scalar_z(
                    result_refine.x
                ),

            "nfev":
                result_da.nfev
                + result_refine.nfev,

            "Success":
                result_refine.success,
        })


    # --------------------------------------------------------
    # Result table
    # --------------------------------------------------------

    for candidate in candidates:

        pars = candidate[
            "pars"
        ]


        row = {
            "Case":
                case_name,

            "Algorithm":
                candidate[
                    "Algorithm"
                ],

            "Run":
                candidate[
                    "Run"
                ],

            "Seed":
                candidate[
                    "Seed"
                ],

            "Objective":
                candidate[
                    "Objective"
                ],

            "HPPC_RMSE_mV":
                candidate[
                    "RMSE_mV"
                ],

            "HPPC_MAE_mV":
                candidate[
                    "MAE_mV"
                ],

            "nfev":
                candidate[
                    "nfev"
                ],

            "Success":
                candidate[
                    "Success"
                ],

            "Bound_hit":
                bound_hit_string(
                    pars,
                    opt_params,
                ),
        }


        for parameter in opt_params:

            row[
                parameter
            ] = pars[
                parameter
            ]


        result_rows.append(
            row
        )


    best = min(
        candidates,
        key=lambda candidate:
            candidate[
                "Objective"
            ],
    )


    return {
        "best":
            best,

        "table":
            pd.DataFrame(
                result_rows
            ),
    }


RESULT_3A = run_optimizer_suite(
    "Stage 3A",
    OPT_BASE_3A,
    OPT_PARAMS_3A,
)


RESULT_3B = run_optimizer_suite(
    "Stage 3B",
    OPT_BASE_3B,
    OPT_PARAMS_3B,
)


BEST_3A = RESULT_3A[
    "best"
]

BEST_3B = RESULT_3B[
    "best"
]


OPTIMIZATION_RUNS = pd.concat(
    [
        RESULT_3A[
            "table"
        ],
        RESULT_3B[
            "table"
        ],
    ],
    ignore_index=True,
)


display(
    OPTIMIZATION_RUNS
    .sort_values(
        [
            "Case",
            "Objective",
        ]
    )
    .round(5)
)


In [ ]:
# ============================================================
# Stage 3 fitting summary
# ============================================================

def best_summary_row(
    case_name,
    best,
    opt_params,
):

    pars = best[
        "pars"
    ]


    row = {
        "Case":
            case_name,

        "Best algorithm":
            best[
                "Algorithm"
            ],

        "Seed":
            best[
                "Seed"
            ],

        "Objective":
            best[
                "Objective"
            ],

        "HPPC_RMSE_mV":
            best[
                "RMSE_mV"
            ],

        "HPPC_MAE_mV":
            best[
                "MAE_mV"
            ],

        "Bound_hit":
            bound_hit_string(
                pars,
                opt_params,
            ),
    }


    for parameter in opt_params:

        row[
            parameter
        ] = pars[
            parameter
        ]


    return row


STAGE3_SUMMARY = pd.DataFrame([
    best_summary_row(
        "Stage 3A",
        BEST_3A,
        OPT_PARAMS_3A,
    ),

    best_summary_row(
        "Stage 3B",
        BEST_3B,
        OPT_PARAMS_3B,
    ),
])


display(
    STAGE3_SUMMARY.round(5)
)


for case_name, best in [
    (
        "Stage 3A",
        BEST_3A,
    ),
    (
        "Stage 3B",
        BEST_3B,
    ),
]:

    metrics = evaluate_hppc_pars(
        best[
            "pars"
        ]
    )

    print(
        "\n===",
        case_name,
        "point-type metrics ==="
    )

    display(
        metrics[
            "Point_table"
        ].round(3)
    )


# Stage 4 — Independent C-rate validation

Stage 4에서는 **HPPC fitting에 사용하지 않은** 아래 6조건만 사용한다.

- 0.5C Charge
- 0.5C Discharge
- 1C Charge
- 1C Discharge
- 2C Charge
- 2C Discharge

비교 모델:
- Stage 1 baseline
- Stage 3A best
- Stage 3B best

평가:
- Full overlapping time RMSE / MAE
- SOC 10–90% time-window RMSE / MAE
- End-time error
- Charge minimum anode potential


In [ ]:
# ============================================================
# Stage 4 independent validation
# ============================================================

def interp_on_time(
    time,
    values,
    grid,
):

    time = np.asarray(
        time,
        dtype=float,
    )

    values = np.asarray(
        values,
        dtype=float,
    )

    valid = (
        np.isfinite(time)
        & np.isfinite(values)
    )

    time = time[valid]
    values = values[valid]

    order = np.argsort(
        time
    )

    time = time[order]
    values = values[order]

    unique_time, index = np.unique(
        time,
        return_index=True,
    )

    unique_values = values[
        index
    ]

    output = np.full_like(
        grid,
        np.nan,
        dtype=float,
    )

    inside = (
        (grid >= unique_time.min())
        & (grid <= unique_time.max())
    )

    output[inside] = np.interp(
        grid[inside],
        unique_time,
        unique_values,
    )

    return output


def validation_metrics(
    experiment,
    simulation,
):

    t_start = max(
        experiment[
            "t_min"
        ].min(),
        simulation[
            "t_min"
        ].min(),
    )

    t_end = min(
        experiment[
            "t_min"
        ].max(),
        simulation[
            "t_min"
        ].max(),
    )

    time_grid = np.linspace(
        t_start,
        t_end,
        400,
    )

    voltage_exp = interp_on_time(
        experiment[
            "t_min"
        ],
        experiment[
            "V"
        ],
        time_grid,
    )

    voltage_model = interp_on_time(
        simulation[
            "t_min"
        ],
        simulation[
            "V"
        ],
        time_grid,
    )

    valid = (
        np.isfinite(
            voltage_exp
        )
        &
        np.isfinite(
            voltage_model
        )
    )

    error_mV = (
        voltage_model[valid]
        - voltage_exp[valid]
    ) * 1000


    soc_mask = (
        (
            experiment[
                "SOC"
            ]
            >= 0.10
        )
        &
        (
            experiment[
                "SOC"
            ]
            <= 0.90
        )
    )


    if (
        np.sum(
            soc_mask
        )
        > 5
    ):

        mid_start = max(
            experiment[
                "t_min"
            ][soc_mask].min(),
            simulation[
                "t_min"
            ].min(),
        )

        mid_end = min(
            experiment[
                "t_min"
            ][soc_mask].max(),
            simulation[
                "t_min"
            ].max(),
        )

        mid_grid = np.linspace(
            mid_start,
            mid_end,
            300,
        )

        voltage_exp_mid = interp_on_time(
            experiment[
                "t_min"
            ],
            experiment[
                "V"
            ],
            mid_grid,
        )

        voltage_model_mid = interp_on_time(
            simulation[
                "t_min"
            ],
            simulation[
                "V"
            ],
            mid_grid,
        )

        valid_mid = (
            np.isfinite(
                voltage_exp_mid
            )
            &
            np.isfinite(
                voltage_model_mid
            )
        )

        error_mid_mV = (
            voltage_model_mid[
                valid_mid
            ]
            - voltage_exp_mid[
                valid_mid
            ]
        ) * 1000

        rmse_mid = np.sqrt(
            np.mean(
                error_mid_mV**2
            )
        )

        mae_mid = np.mean(
            np.abs(
                error_mid_mV
            )
        )

    else:

        rmse_mid = np.nan
        mae_mid = np.nan


    return {
        "RMSE_full_mV":
            np.sqrt(
                np.mean(
                    error_mV**2
                )
            ),

        "MAE_full_mV":
            np.mean(
                np.abs(
                    error_mV
                )
            ),

        "RMSE_SOC10_90_mV":
            rmse_mid,

        "MAE_SOC10_90_mV":
            mae_mid,

        "End_time_error_min":
            (
                simulation[
                    "t_min"
                ][-1]
                - experiment[
                    "t_min"
                ][-1]
            ),
    }


VALIDATION_CASES = {
    "Stage 1":
        PARS_NOMINAL,

    "Stage 3A":
        BEST_3A[
            "pars"
        ],

    "Stage 3B":
        BEST_3B[
            "pars"
        ],
}


VALIDATION_SIMS = {}
validation_rows = []


for model_name, pars in (
    VALIDATION_CASES.items()
):

    for rate in [
        0.5,
        1.0,
        2.0,
    ]:

        for charge in [
            True,
            False,
        ]:

            direction = (
                "Charge"
                if charge
                else "Discharge"
            )

            print(
                model_name,
                rate,
                "C",
                direction,
            )


            try:

                simulation = run_cc_dfn(
                    rate,
                    charge,
                    pars,
                )


                VALIDATION_SIMS[
                    (
                        model_name,
                        rate,
                        charge,
                    )
                ] = simulation


                experiment = EXP_DYN[
                    (
                        rate,
                        charge
                    )
                ]


                metrics = validation_metrics(
                    experiment,
                    simulation,
                )


                min_anode = np.nan
                soc_at_min = np.nan
                time_at_min = np.nan


                if charge:

                    min_index = int(
                        np.nanargmin(
                            simulation[
                                "Anode_potential_V"
                            ]
                        )
                    )

                    min_anode = float(
                        simulation[
                            "Anode_potential_V"
                        ][
                            min_index
                        ]
                    )

                    soc_at_min = float(
                        simulation[
                            "SOC"
                        ][
                            min_index
                        ]
                    )

                    time_at_min = float(
                        simulation[
                            "t_min"
                        ][
                            min_index
                        ]
                    )


                validation_rows.append({
                    "Model":
                        model_name,

                    "C-rate":
                        rate,

                    "Direction":
                        direction,

                    **metrics,

                    "Min_anode_V":
                        min_anode,

                    "SOC_at_min_anode":
                        soc_at_min,

                    "Time_at_min_anode_min":
                        time_at_min,

                    "Status":
                        "OK",
                })


            except Exception as error:

                print(
                    "FAILED:",
                    error
                )


                validation_rows.append({
                    "Model":
                        model_name,

                    "C-rate":
                        rate,

                    "Direction":
                        direction,

                    "RMSE_full_mV":
                        np.nan,

                    "MAE_full_mV":
                        np.nan,

                    "RMSE_SOC10_90_mV":
                        np.nan,

                    "MAE_SOC10_90_mV":
                        np.nan,

                    "End_time_error_min":
                        np.nan,

                    "Min_anode_V":
                        np.nan,

                    "SOC_at_min_anode":
                        np.nan,

                    "Time_at_min_anode_min":
                        np.nan,

                    "Status":
                        "FAIL",
                })


VALIDATION_TABLE = pd.DataFrame(
    validation_rows
)


display(
    VALIDATION_TABLE.round(4)
)


In [ ]:
# ============================================================
# Stage 4 plots
# ============================================================

fig, axes = plt.subplots(
    2,
    3,
    figsize=(14, 7.5),
    sharey=True,
)


for col, rate in enumerate(
    [0.5, 1.0, 2.0]
):

    for row, charge in enumerate(
        [True, False]
    ):

        ax = axes[
            row,
            col
        ]

        experiment = EXP_DYN[
            (
                rate,
                charge
            )
        ]


        ax.plot(
            experiment[
                "t_min"
            ],
            experiment[
                "V"
            ],
            lw=2.5,
            label="Experiment",
        )


        for model_name in (
            VALIDATION_CASES
        ):

            key = (
                model_name,
                rate,
                charge,
            )

            if (
                key
                not in
                VALIDATION_SIMS
            ):
                continue


            simulation = (
                VALIDATION_SIMS[
                    key
                ]
            )


            ax.plot(
                simulation[
                    "t_min"
                ],
                simulation[
                    "V"
                ],
                lw=1.7,
                label=model_name,
            )


        ax.set_title(
            f"{rate}C "
            f"{'Charge' if charge else 'Discharge'}"
        )

        ax.set_xlabel(
            "Time [min]"
        )

        ax.set_ylabel(
            "Terminal voltage [V]"
        )

        ax.legend(
            fontsize=8
        )


plt.tight_layout()
plt.show()


fig, axes = plt.subplots(
    1,
    3,
    figsize=(14, 4.2),
)


for col, rate in enumerate(
    [0.5, 1.0, 2.0]
):

    ax = axes[col]


    for model_name in (
        VALIDATION_CASES
    ):

        key = (
            model_name,
            rate,
            True,
        )

        if (
            key
            not in
            VALIDATION_SIMS
        ):
            continue


        simulation = (
            VALIDATION_SIMS[
                key
            ]
        )


        ax.plot(
            simulation[
                "t_min"
            ],
            simulation[
                "Anode_potential_V"
            ],
            lw=1.8,
            label=model_name,
        )


    ax.axhline(
        0,
        ls="--",
        lw=1,
    )

    ax.set_title(
        f"{rate}C Charge"
    )

    ax.set_xlabel(
        "Time [min]"
    )

    ax.set_ylabel(
        "Anode potential [V]"
    )

    ax.legend(
        fontsize=8
    )


plt.tight_layout()
plt.show()


## 해석 순서

1. **Stage 1**
   - Endpoint error가 충분히 작은지
   - qOCV RMSE / MAE 기록

2. **Stage 2A / 2B**
   - sensitivity magnitude
   - cosine similarity
   - SVD / condition number
   - `AUTO_SUBSET_7`, `AUTO_SUBSET_4`가 물리적으로 납득 가능한지 확인
   - 필요하면 Stage 3 실행 전에 `OPT_PARAMS_3A`, `OPT_PARAMS_3B`를 수동 수정

3. **Stage 3**
   - 최저 HPPC objective 하나만 보지 말 것
   - 서로 다른 optimizer / seed가 비슷한 parameter 영역에 수렴하는지 확인
   - bound hit 여부 확인
   - HPPC RMSE / MAE와 point-type별 error 함께 확인

4. **Stage 4**
   - HPPC fitting에 사용하지 않은 0.5C / 1C / 2C Charge·Discharge에서 일반화 성능 확인
   - Stage 3A / 3B 중 RMSE/MAE뿐 아니라 parameter 안정성까지 함께 비교
   - anode potential은 최종 절대값이 아니라 내부상태 prediction으로 추적
